<a href="https://colab.research.google.com/github/dianaalme46-hue/compling-course/blob/main/fine_tuning_HW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [ ]:
!pip install transformers datasets evaluate accelerate gradio -q #гредио поможет делать интерфейсы
!pip install huggingface_hub -q
import torch
print(f"GPU доступен: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Тип GPU: {torch.cuda.get_device_name(0)}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00
GPU доступен: True
Тип GPU: Tesla T4


In [ ]:
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from datasets import load_dataset
import evaluate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
#Загрузить датасет
dataset = load_dataset('ag_news')
print(f"Датасет загружен. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Датасет загружен. Train: 120000, Test: 7600


In [ ]:
#Модель для обучения и токенайзер
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4
).to(device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#Определение функции токенизации
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)
tokenized_datasets = dataset.map(tokenize_function, batched=True)
train_dataset = tokenized_datasets["train"]
eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(5000)) #обработать только 5к примеров

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
# Настройка обучения
training_args = TrainingArguments(
    output_dir="./results-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    logging_steps=500,
)

In [ ]:
# Метрика и обучение
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
print("Training on AG_news...")
trainer.train()

Training on AG_news...


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
#Оценка
ag_results = trainer.evaluate() # получите метрики оценки
print(f"AG News results: {ag_results}")


Epoch,Training Loss,Validation Loss,Accuracy
0,0.265393,0.268670,0.911600


AG News results: {'eval_loss': 0.2686699330806732, 'eval_accuracy': 0.9116}


In [ ]:
from transformers import pipeline
classifier = pipeline('text-classification', model=model, tokenizer=tokenizer)
test_texts = [
    'Team USA’s Oksana Masters claims sprint gold in the women’s sitting event as Liu Zixu, Vilde Nilsen, and Raman Svirydzenka also triumph on a busy day of Para cross-country skiing at Milano Cortina 2026.',
    'Oil prices have surged this week, raising the price at the pump just as Americans are already struggling with affordability.',
    'China’s top diplomat cast his country as a defender of peace and stability as war in Iran rages, while striking a conciliatory tone towards the United States ahead of a highly anticipated summit between the two nation’s leaders.'
]
for text in test_texts:
    result = classifier(text)[0]
    print(f"Text: {text}\nClassification: {result['label']}, Score: {result['score']:.4f}\n")

Text: Team USA’s Oksana Masters claims sprint gold in the women’s sitting event as Liu Zixu, Vilde Nilsen, and Raman Svirydzenka also triumph on a busy day of Para cross-country skiing at Milano Cortina 2026.
Classification: LABEL_1, Score: 0.9889

Text: Oil prices have surged this week, raising the price at the pump just as Americans are already struggling with affordability.
Classification: LABEL_2, Score: 0.9798

Text: China’s top diplomat cast his country as a defender of peace and stability as war in Iran rages, while striking a conciliatory tone towards the United States ahead of a highly anticipated summit between the two nation’s leaders.
Classification: LABEL_0, Score: 0.9953

